## Run test: Qwen/Qwen3-VL-8B-Instruct

Aligned with the Hugging Face generated direct-load/Colab-style snippet where possible. Inference Providers snippets are ignored because they do not validate local AMD GPU loading. The direct path uses AutoModelForMultimodalLM; the README also supports the concrete Qwen3VLForConditionalGeneration class.

In [ ]:
%pip install -U transformers accelerate

In [ ]:
# HF Colab aligned image-text-to-text pipeline smoke test on one visible GPU.
import gc
import torch
from transformers import pipeline

model_path = "Qwen/Qwen3-VL-8B-Instruct"

messages = [{'role': 'user',
  'content': [{'type': 'image',
               'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG'},
              {'type': 'text', 'text': 'What animal is on the candy?'}]}]
pipe = pipeline("image-text-to-text", model=model_path, device=0, trust_remote_code=True)
pipe_result = pipe(messages, max_new_tokens=40)
print(pipe_result)
pipe_tensor_devices = sorted({str(p.device) for p in pipe.model.parameters()})
print("pipe parameter devices =", pipe_tensor_devices)

del pipe
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# HF Colab aligned direct image inference smoke test.
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForMultimodalLM.from_pretrained(model_path).to("cuda").eval()

messages = [{'role': 'user',
  'content': [{'type': 'image',
               'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG'},
              {'type': 'text', 'text': 'What animal is on the candy?'}]}]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
